<h1 align='center'>Synthetic Data Generation and Unsloth Tutorial</h1>

## 📚 Table of Contents:

- [Synthetic Data Kit: Data Generation](#synthetic-data-generation)
- [Unsloth: Fine-Tuning and saving the model](#fine-tuning)

## Synthetic Data Generation

In this section, we use the CLI from synthetic-data-kit to generate datasets

### Testing Synthetic Data Kit Command

Please make sure you are running vllm by opening a terminal and typing `vllm serve Unsloth/Llama-3.3-70B-Instruct   --port 8001   --max-model-len 48000   --gpu-memory-utilization 0.85`

### Exploring Synthetic Data Kit CLI

This command displays the help menu for the `synthetic-data-kit` CLI tool, showing available commands:
- **system-check**: Verify LLM provider server is running
- **ingest**: Parse documents (PDF, HTML, YouTube, etc.) into clean text
- **create**: Generate synthetic content (Q&A pairs, instructions, etc.) using LLM
- **curate**: Filter and clean generated content based on quality scores
- **save-as**: Convert data to different formats (fine-tuning format, JSON, etc.)
- **server**: Launch web interface for the toolkit

In [1]:
!synthetic-data-kit --help

Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
                                                                                
 Usage: synthetic-data-kit [OPTIONS] COMMAND [ARGS]...                          
                                                                                
 A toolkit for preparing synthetic datasets for fine-tuning LLMs                
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --config              -c      PATH  Path to configuration file               │
│ --install-completion                Install completion for the current       │
│                                     shell.                                  

### Verifying LLM Server Status

This command checks if the vLLM server is running and accessible at `http://localhost:8001/v1`. It displays:
- Server status and endpoint
- Available models (here: Unsloth/Llama-3.3-70B-Instruct)
- Model configuration (max context length: 48000 tokens)

The system is configured to use the vLLM provider as specified in `tutorial_config.yaml`.

In [2]:
!synthetic-data-kit -c tutorial_config.yaml system-check

Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: tutorial_config.yaml
Config has LLM provider set to: vllm
Environment variable check:
API_ENDPOINT_KEY: Not found
get_llm_provider returning: vllm
 vLLM server is running at http://localhost:8001/v1
Available models: {'object': 'list', 'data': [{'id': 
'Qwen/Qwen2.5-14B-Instruct', 'object': 'model', 'created': 1771137032, 
'owned_by': 'vllm', 'root': 'Qwen/Qwen2.5-14B-Instruct', 'parent': None, 
'max_model_len': 32768, 'permission': [{'id': 
'modelperm-7bd1e840305b47bb9df7151e43f262ee', 'object': 'model_permission', 
'created': 1771137032, 'allow_create_engine': False, 'allow_sampling': True, 
'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 
'allow_fine_tuning': False, 

### Creating Project Directory Structure

This command creates a well-organized directory structure for the logical reasoning project:
- `sources/`: Store original source documents (PDFs, etc.)
- `data/input/`: Input files for processing
- `data/parsed/`: Parsed text files after document ingestion
- `data/generated/`: Generated synthetic Q&A pairs
- `data/curated/`: Quality-filtered data after curation
- `data/final/`: Final formatted data ready for fine-tuning

In [3]:
!bash -c 'mkdir -p logical_reasoning/{sources,data/{input,parsed,generated,curated,final}}'

### Navigating to Project Directory

Changes the current working directory to `logical_reasoning/` where all subsequent operations will take place.

In [4]:
cd logical_reasoning

/workspace/AAIPL/logical_reasoning


### Downloading Source Documents

Downloads two PDF documents related to logical reasoning and liar/truth puzzles:
1. "Logical Reasoning" textbook from CSU Sacramento
2. "Liar and Truth Teller Puzzles" from UMass

These documents will serve as the knowledge base for generating synthetic training data. The `-q` flag runs wget in quiet mode, and `--show-progress` displays a progress bar.

In [5]:
!wget -P sources/ -q --show-progress   "https://www.csus.edu/faculty/d/dowden/_internal/_documents/logical-reasoning-12.pdf"   "https://people.cs.umass.edu/~pthomas/solutions/Liar_Truth.pdf"

logical-reasoning-1 100%[===================>]   5.52M  1.55MB/s    in 3.7s    
Liar_Truth.pdf.1    100%[===================>] 327.61K  --.-KB/s    in 0.1s    


### Copying Source Files to Input Directory

Copies all downloaded source documents from `sources/` to `data/input/` to prepare them for the ingestion pipeline.

In [6]:
!cp sources/* data/input/

### Ingesting and Parsing Documents

This command processes the PDF files in `data/input/` using the synthetic-data-kit's **ingest** command:
- Extracts text content from PDFs
- Cleans and normalizes the text
- Saves parsed text files to `data/parsed/`

The output shows successful processing of 2 PDF files (Liar_Truth.pdf and logical-reasoning-12.pdf).

In [7]:
!synthetic-data-kit ingest ./data/input/

Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Processing directory: ./data/input/
Found 2 supported files to process
✓ Liar_Truth.pdf
✓ logical-reasoning-12.pdf

Processing Summary:
Total files: 2
Successful: 2
Failed: 0
✅ All files processed successfully!


### Generating Synthetic Q&A Pairs

This command uses the synthetic-data-kit's **create** command to generate Q&A pairs from the parsed text:
- Reads parsed text files from `data/parsed/`
- Uses the vLLM provider with Llama-3.3-70B-Instruct model
- Generates 50 Q&A pairs per file (`--num-pairs 50`)
- Type is set to `qa` for question-answer pair generation
- Outputs are saved to `data/generated/`

The process chunks the text and generates questions with corresponding answers. This took about 10 minutes for the full run. Use `--verbose` flag to see detailed progress or reduce `--num-pairs` for faster testing.

Note: This will take about 10 minutes, set `--verbose` flag to see progress or reduce the `num-pairs` for a faster test

In [8]:
!synthetic-data-kit -c ../tutorial_config.yaml create ./data/parsed/ --type qa --num-pairs 50

Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: ../tutorial_config.yaml
Config has LLM provider set to: vllm
get_llm_provider returning: vllm
🔗 Using vllm provider
Processing directory: ./data/parsed/ for qa generation
Found 2 qa files to process
Loading config from: ../tutorial_config.yaml
Config has LLM provider set to: vllm
L Using vllm provider
Loading config from: ../tutorial_config.yaml
Config has LLM provider set to: vllm
Processing 1 chunks to generate QA pairs...
Batch processing complete.                                                      
Generated 24 QA pairs total (requested: 50)
Saving result to data/generated/Liar_Truth_qa_pairs.json
Successfully wrote test file to data/generated/test_write.json
Successfully wrote result to da

### Curating and Quality Filtering

This command uses the **curate** function to filter generated Q&A pairs based on quality:
- Evaluates each Q&A pair using quality metrics
- Filters pairs with quality score above threshold (7.0/10)
- Removes low-quality, inconsistent, or malformed pairs
- Saves curated data to `data/curated/`

This ensures only high-quality synthetic data is used for fine-tuning.

Note: This will also take about 10 minutes.

In [9]:
!synthetic-data-kit -c ../tutorial_config.yaml curate ./data/generated/ --threshold 7.0

Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: ../tutorial_config.yaml
Config has LLM provider set to: vllm
get_llm_provider returning: vllm
🔗 Using vllm provider
Processing directory: ./data/generated/ for curation
Found 3 JSON files to curate
Loading config from: ../tutorial_config.yaml
Config has LLM provider set to: vllm
Loading config from: ../tutorial_config.yaml
Config has LLM provider set to: vllm
Processing 5 batches of QA pairs...
Batch processing complete.                                                      
Rated 24 QA pairs
Retained 20 pairs (threshold: 7.0)
Average score: 7.4
✓ Liar_Truth_qa_pairs.json
Loading config from: ../tutorial_config.yaml
Config has LLM provider set to: vllm
Loading config from: ../tutorial_config.yaml


### Converting to Fine-Tuning Format

This command uses the **save-as** function to convert curated Q&A pairs to fine-tuning format:
- Reads curated JSON files from `data/curated/`
- Converts to format `ft` (fine-tuning format with messages structure)
- Outputs are saved to `data/final/` with proper conversation format
- The resulting format is compatible with standard fine-tuning pipelines

Successfully converted 2 files to fine-tuning format.

In [10]:
!synthetic-data-kit save-as ./data/curated/ --format ft

Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Loading config from: /usr/local/lib/python3.12/dist-packages/synthetic_data_kit/config.yaml
Config has LLM provider set to: api-endpoint
Processing directory: ./data/curated/ for format conversion to ft
Found 2 JSON files to convert to ft format
✓ Liar_Truth_qa_pairs_cleaned.json
✓ logical-reasoning-12_qa_pairs_cleaned.json

Format Conversion Summary (ft, json):
Total files: 2
Successful: 2
Failed: 0
✅ All files converted successfully!


### Loading and Converting Data to HuggingFace Dataset

This cell performs comprehensive data processing:

1. **Finding Files**: Locates all JSON files in `data/final/` directory
2. **Loading Data**: Reads each JSON file containing fine-tuning formatted data
3. **Format Conversion**: Extracts user and assistant messages from the fine-tuning format
4. **Structuring Conversations**: Creates a standardized conversation format with role-content pairs
5. **Creating Dataset**: Converts the processed data into a HuggingFace Dataset object

The output shows 74 total conversations were successfully loaded and formatted. The preview displays a sample conversation showing either a knight-and-knave logic puzzle with its solution or a deduction puzzle about the Logical Reasoning book.

In [11]:
import json
import glob
from pathlib import Path
from datasets import Dataset

# ===== CONFIGURATION =====
data_dir = "./data/final"  # Change this to your data directory

# ===== STEP 1: Find all FT files =====
data_path = Path(data_dir)
ft_files = glob.glob(str(data_path / "*.json"))

# ===== STEP 2: Load and convert all files =====
all_data = []

for file_path in ft_files:
    # Load the JSON file
    with open(file_path, 'r') as f:
        ft_data = json.load(f)
    
    # Convert each item
    for item in ft_data:
        if 'messages' not in item:
            continue
        
        # Extract only user and assistant messages
        conversation = []
        for msg in item['messages']:
            if msg['role'] == 'user' or msg['role'] == 'assistant':
                conversation.append({
                    "role": msg['role'],
                    "content": msg['content']
                })
        
        # Add to our data if we have at least one exchange
        if len(conversation) > 0:
            all_data.append({
                "conversations": conversation
            })

print(f"\n🎯 Total conversations: {len(all_data)}")

# ===== STEP 3: Create HuggingFace Dataset =====
dataset = Dataset.from_list(all_data)

# ===== STEP 4: Preview the data =====
print(json.dumps(dataset[0], indent=2))


🎯 Total conversations: 55
{
  "conversations": [
    {
      "content": "In a small island kingdom, there are two types of inhabitants: knights who always tell the truth and knaves who always lie. You encounter three inhabitants, A, B, and C. A says, 'B and C are both knaves.' B says, 'Exactly one of us is a knight.' C remains silent. Who are A, B, and C?",
      "role": "user"
    },
    {
      "content": "To determine the identities of A, B, and C, we need to analyze the statements made by A and B. Step 1: Analyze A's statement. A says, 'B and C are both knaves.' If A were a knight, then B and C would indeed both be knaves, but this creates a contradiction because if B were a knave, B\u2019s statement would be false, meaning that either both or none of A, B, and C are knights, which contradicts A being a knight. Therefore, A must be a knave, meaning B and C cannot both be knaves. Step 2: Analyze B's statement. B says, 'Exactly one of us is a knight.' Since we determined A is a knav

## Fine-Tuning

### Note: Please remember to shutdown the vLLM instance!

### Importing Standard Libraries

Imports essential Python libraries for fine-tuning:
- `os`, `json`, `glob`: File system operations and JSON handling
- `torch`: PyTorch deep learning framework
- `shutil`: File operations
- `Path`: Path manipulation
- `Dataset`: HuggingFace datasets library for data handling

In [12]:
import os
import json
import glob
import torch
import shutil
from pathlib import Path
from datasets import Dataset

### Importing Unsloth and Training Libraries

Imports specialized libraries for efficient fine-tuning:
- `FastLanguageModel` from Unsloth: Optimized model loading and training
- `get_chat_template`, `standardize_sharegpt`, `train_on_responses_only`: Chat formatting utilities
- `SFTConfig`, `SFTTrainer`: Supervised fine-tuning configuration and trainer from TRL
- `DataCollatorForSeq2Seq`: Handles batching and padding for sequence-to-sequence training

In [13]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, standardize_sharegpt, train_on_responses_only
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
#### Unsloth: `hf_xet==1.1.10` and `ipykernel>6.30.1` breaks progress bars. Disabling for now in XET.
#### Unsloth: To re-enable progress bars, please downgrade to `ipykernel==6.30.1` or wait for a fix to
https://github.com/huggingface/xet-core/issues/526
INFO 02-15 06:35:12 [__init__.py:225] Automatically detected platform rocm.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Setup Unsloth model and tokenizer for ROCm without bitsandbytes

### Loading Llama-3.3-70B Model with LoRA

This cell sets up the model for efficient fine-tuning on AMD ROCm hardware:

**Model Configuration:**
- Model: Llama-3.3-70B-Instruct (70 billion parameters)
- Data type: bfloat16 for ROCm compatibility
- No quantization (load_in_4bit=False) to avoid bitsandbytes dependency
- Max sequence length: 1024 tokens

**LoRA (Low-Rank Adaptation) Configuration:**
- Rank (r): 64 - Higher rank for the large 70B model
- Target modules: All attention and MLP layers (q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj)
- LoRA alpha: 64
- Dropout: 0 (no dropout)
- Gradient checkpointing: "unsloth" for memory efficiency

LoRA enables efficient fine-tuning by only training small adapter layers instead of the entire 70B model, making it feasible to train on a single AMD MI300X GPU with 192GB HBM3 memory.

In [17]:
import os
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
os.environ['HF_DATASETS_CACHE'] = '/tmp/hf_cache'

from huggingface_hub import snapshot_download

print("📥 Downloading Qwen/Qwen2.5-14B-Instruct to /tmp/hf_cache...")
snapshot_download(
    repo_id="Qwen/Qwen2.5-14B-Instruct",
    cache_dir="/tmp/hf_cache",
    local_dir="/tmp/hf_cache/models--Qwen--Qwen2.5-14B-Instruct/snapshots/main",
    local_dir_use_symlinks=False
)
print("✅ Model downloaded")

📥 Downloading Qwen/Qwen2.5-14B-Instruct to /tmp/hf_cache...
✅ Model downloaded


In [25]:
import os
import torch
from unsloth import FastLanguageModel

# 1. Force cache paths explicitly
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'

# 2. DEFINITIVE FIX: Use the ABSOLUTE PATH to the downloaded model
# This tells Unsloth exactly where the files are, bypassing lookup/download errors
model_path = "/tmp/hf_cache/models--Qwen--Qwen2.5-14B-Instruct/snapshots/main"

print(f"📂 Loading model from local path: {model_path}")

max_seq_length = 1024
dtype = torch.bfloat16
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,  # <--- pointing to local folder
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = "auto",
    torch_dtype = torch.bfloat16,
    trust_remote_code = True,
)

print("✅ Model loaded successfully!")

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

📂 Loading model from local path: /tmp/hf_cache/models--Qwen--Qwen2.5-14B-Instruct/snapshots/main
Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.10.9: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


[2026-02-15 06:52:54] INFO modeling.py:987: We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Model loaded successfully!


### Preparing Dataset with Chat Template

This cell formats the dataset for fine-tuning:

**Steps:**
1. **Set Chat Template**: Applies Llama-3.1 chat template formatting
2. **Configure Padding**: Sets pad token to eos token if not already set
3. **Format Conversations**: The `formatting_prompts_func` function:
   - Takes raw conversations from the dataset
   - Applies the chat template to format them properly
   - Validates conversation structure (list of dicts with role/content)
   - Filters out malformed conversations
4. **Standardize Format**: Uses `standardize_sharegpt` to normalize the data structure
5. **Apply Formatting**: Maps the formatting function across all examples
6. **Remove Empty**: Filters out any empty or invalid formatted texts

The output shows 74 valid examples were successfully prepared. A sample of the formatted text is displayed, showing the proper Llama-3.1 chat template structure with system, user, and assistant headers.

In [26]:
"""Prepare dataset with proper chat template and tensor compatibility"""
print("🔧 Preparing dataset for training...")

# Set chat template (Qwen uses ChatML)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Formatting function that ensures proper tensor conversion
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = []
    
    for convo in convos:
        # Ensure conversation is in correct format
        if isinstance(convo, list) and all(isinstance(msg, dict) for msg in convo):
            text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        else:
            print(f"⚠️  Skipping malformed conversation: {type(convo)}")
            continue
    
    return {"text": texts}

dataset = standardize_sharegpt(dataset)

dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

print(f"✅ Prepared {len(dataset)} valid examples for training")

# Show sample
if len(dataset) > 0:
    print(f"📝 Sample formatted text:")
    print(dataset["text"][0][:200] + "...")

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


🔧 Preparing dataset for training...


num_proc must be <= 55. Reducing num_proc to 55 for dataset of size 55.
[2026-02-15 06:53:35] WARNING arrow_dataset.py:3114: num_proc must be <= 55. Reducing num_proc to 55 for dataset of size 55.


Unsloth: Standardizing formats (num_proc=55):   0%|          | 0/55 [00:00<?, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

Filter:   0%|          | 0/55 [00:00<?, ? examples/s]

✅ Prepared 55 valid examples for training
📝 Sample formatted text:
<|im_start|>user
In a small island kingdom, there are two types of inhabitants: knights who always tell the truth and knaves who always lie. You encounter three inhabitants, A, B, and C. A says, 'B an...


### Training the Model with ROCm-Optimized Settings

This cell configures and executes the fine-tuning process:

**Training Configuration (SFTConfig):**
- **Batch size**: 64 per device - leveraging the AMD MI300X's massive 192GB HBM3 memory
- **Gradient accumulation**: 1 step
- **Warmup**: 5 steps
- **Epochs**: 1 full pass through the dataset
- **Learning rate**: 1e-4
- **Optimizer**: adamw_8bit for memory efficiency
- **Precision**: bf16 (bfloat16) for ROCm
- **Gradient checkpointing**: Enabled for memory efficiency

**Special Training Mode:**
Uses `train_on_responses_only` to compute loss only on the assistant's responses, not on the user's questions. This focuses the model on learning to generate accurate answers rather than memorizing the input format.

**Key Features:**
- DataCollatorForSeq2Seq handles variable-length sequences with proper padding
- No packing to preserve conversation structure
- Single dataloader worker for ROCm stability
- Gradient checkpointing via Unsloth for memory optimization

The model is then trained on the 74 logical reasoning conversations.

In [27]:
"""Train model with ROCm-optimized settings"""
# Ensure tokenizer has proper padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Setup trainer with ROCm-friendly settings and proper data handling
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=32,  # 🚀 MI300X can handle this with 192GB HBM3!
        gradient_accumulation_steps=2,   # Effective batch size = 32*2 = 64
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=1e-4,
        logging_steps=1,
        optim="adamw_8bit",  # Pure torch optimizer
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="logical_reasoning_rocm_outputs",
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        remove_unused_columns=True,  # Remove unused columns to avoid tensor issues
        gradient_checkpointing=True,
        dataloader_num_workers=0,  # Single worker for ROCm stability
    ),
)

# Train only on responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

FastLanguageModel.for_training(model)
trainer_stats = trainer.train()


trainer_stats = trainer.train()

num_proc must be <= 55. Reducing num_proc to 55 for dataset of size 55.
[2026-02-15 06:53:55] WARNING arrow_dataset.py:3114: num_proc must be <= 55. Reducing num_proc to 55 for dataset of size 55.


Unsloth: Tokenizing ["text"] (num_proc=55):   0%|          | 0/55 [00:00<?, ? examples/s]

num_proc must be <= 55. Reducing num_proc to 55 for dataset of size 55.
[2026-02-15 06:54:09] WARNING arrow_dataset.py:3114: num_proc must be <= 55. Reducing num_proc to 55 for dataset of size 55.


Map (num_proc=55):   0%|          | 0/55 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 55 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 137,625,600 of 14,907,659,264 (0.92% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.993700


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 55 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 137,625,600 of 14,907,659,264 (0.92% trained)


Step,Training Loss
1,0.993700


### Saving the Fine-Tuned Model

This cell saves the trained model in two formats:

1. **LoRA Adapters** (`logical_reasoning_rocm_lora/`):
   - Saves only the trained LoRA adapter weights (lightweight, ~few hundred MB)
   - Can be loaded later with the base model
   - Useful for sharing or deploying with the original base model

2. **Merged Model** (`logical_reasoning_rocm_merged/`):
   - Merges LoRA adapters back into the base model
   - Creates a standalone model with all weights
   - Saved in 16-bit precision for better quality
   - Ready for immediate inference without loading adapters

Both formats include the tokenizer configuration. The merged model is production-ready and can be used directly for generating answers to logical reasoning questions.

In [28]:
"""Save the trained model"""
print("\n💾 SAVING ROCM-TRAINED MODEL")

# Save LoRA adapters
lora_path = "logical_reasoning_rocm_lora"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA adapters saved to: {lora_path}")

# Save merged model
merged_path = "logical_reasoning_rocm_merged"
print("🔄 Saving merged model...")
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
print(f"✅ Merged model saved to: {merged_path}")

print(f"\n🎉 ROCM MODEL READY!")


💾 SAVING ROCM-TRAINED MODEL
✅ LoRA adapters saved to: logical_reasoning_rocm_lora
🔄 Saving merged model...
Detected local model directory: /tmp/hf_cache/models--Qwen--Qwen2.5-14B-Instruct/snapshots/main
Found HuggingFace hub cache directory: /tmp/hf_cache/hub


Unsloth: Preparing safetensor model files:  12%|█▎        | 1/8 [00:00<00:05,  1.17it/s]

Copied model-00008-of-00008.safetensors from local model directory


Unsloth: Preparing safetensor model files:  25%|██▌       | 2/8 [00:02<00:09,  1.56s/it]

Copied model-00006-of-00008.safetensors from local model directory


...

Copied model-00002-of-00008.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████| 8/8 [00:15<00:00,  1.96s/it]


Copied model-00004-of-00008.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 8/8 [00:50<00:00,  6.37s/it]

Unsloth: Merge process complete. Saved to `/workspace/AAIPL/logical_reasoning/logical_reasoning_rocm_merged`
✅ Merged model saved to: logical_reasoning_rocm_merged

🎉 ROCM MODEL READY!


### Testing the Fine-Tuned Model
You can do a quick test to check that the model is working.

In [29]:
"""Test the fine-tuned model with inference"""
# Switch model to inference mode
FastLanguageModel.for_inference(model)

# Test question - a classic knight/knave logic puzzle
test_question = "A says 'B is a knave.' B says 'A and I are different types.' What are A and B?"

# Format the prompt using the chat template
messages = [
    {"role": "user", "content": test_question}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

# Decode and print
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*70)
print("🧪 TEST INFERENCE")
print("="*70)
print(f"Question: {test_question}")
print(f"\nAnswer:\n{response}")
print("="*70)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



🧪 TEST INFERENCE
Question: A says 'B is a knave.' B says 'A and I are different types.' What are A and B?

Answer:
user
A says 'B is a knave.' B says 'A and I are different types.' What are A and B?
assistant
To solve this problem, we need to determine the truthfulness of statements made by A and B, considering that each person is either a knight (always tells the truth) or a knave (always lies).

Let's analyze the statements step-by-step:

1. **Statement from A**: "B is a knave."
   - If A is a knight, then A's statement must be true, meaning B is indeed a knave.
   - If A is a knave, then A's statement must be false, meaning B is actually a knight.

2. **Statement from B**: "A and I are different types."
   - If B is a knight, then B's statement must be true, meaning one of them (A or B) is a knight and the other is a knave.
   - If B is a knave, then B's statement must be false, meaning they are the same type (both knights or both knaves), which contradicts the definition of knaves

In [ ]:
#fin